<a href="https://colab.research.google.com/github/winarwahyuw/data-science-250401020025-winar-wahyu-wulansari/blob/main/Data_Science_Pertemuan_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
# ================================================
# PERTEMUAN 3: PERSIAPAN & MEMUAT DATASET
# ================================================
# Nama : Winar Wahyu Wulansari
# NIM : 250401020025
# Kelas : IF405
# ================================================


In [22]:
import pandas as pd
import numpy as np
import requests
from pandas import json_normalize
from scipy.stats.mstats import winsorize
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Path file di Google Drive
file_path = '/content/drive/MyDrive/Colab Notebooks/housing_dirty.csv'

try:
    df = pd.read_csv(file_path)
    print("--- [SUKSES] Dataset berhasil dimuat ---")
    print(df.head())
except FileNotFoundError:
    print(f"--- [ERROR] File tidak ditemukan: {file_path} ---")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- [SUKSES] Dataset berhasil dimuat ---
   id  luas_m2  harga_juta   kota  kamar  tahun_bangun kondisi
0   1    297.0      1084.0  jogja    2.0          2000    baik
1   2    254.0       761.0  Medan    NaN          1995   Bagus
2   3    249.7       895.0  Depok    NaN          1983    baik
3   4     49.7       178.0    YGY    5.0          2013    baik
4   5    133.4       424.0  Medan    5.0          2004  Sedang


In [23]:
print("\n=== EKSPLORASI AWAL ===")
print("\n--- Info Dataset ---")
print(df.info())
print("\n--- Statistik Deskriptif ---")
print(df.describe())
print("\n--- Jumlah Missing Values Per Kolom ---")
print(df.isnull().sum())


=== EKSPLORASI AWAL ===

--- Info Dataset ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None

--- Statistik Deskriptif ---
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33

In [24]:
# Hapus Baris Duplikat
# ------------------------
print("\n=== MENANGANI DUPLIKAT ===")
print('Shape awal:', df.shape)
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)


=== MENANGANI DUPLIKAT ===
Shape awal: (130, 7)
Setelah hapus duplikat: (130, 7)


In [25]:
print("\n=== LANGKAH 6: NORMALISASI STRING ===")
# Menggunakan .str.strip().str.title() untuk kota dan .str.lower() untuk kondisi
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()
print("Normalisasi string selesai.")


=== LANGKAH 6: NORMALISASI STRING ===
Normalisasi string selesai.


In [26]:
print("\n=== IMPUTASI MISSING VALUES ===")
# Menggunakan median untuk numerik, modus untuk kategorik
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])
print("Imputasi missing values selesai.")


=== IMPUTASI MISSING VALUES ===
Imputasi missing values selesai.


In [27]:
print("\n=== PENANGANAN OUTLIER ===")
# Melakukan clipping batas IQR untuk kolom harga_juta, luas_m2, dan tahun_bangun [cite: 234]
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
  Q1, Q3 = df[col].quantile([0.25, 0.75])
  IQR = Q3 - Q1
  df[col] = df[col].clip(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
print("Penanganan outlier dengan IQR Fence selesai.")


=== PENANGANAN OUTLIER ===
Penanganan outlier dengan IQR Fence selesai.


In [28]:
print("\n=== VALIDASI DAN EKSPOR ===")
# Memastikan tidak ada missing values dan duplikat yang tersisa
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)

# Menyimpan file bersih ke Google Drive/lokal Colab
output_path = '/content/drive/MyDrive/Colab Notebooks/housing_clean.csv'
df.to_csv(output_path, index=False)
print(f"Dataset bersih tersimpan di: {output_path}")


=== VALIDASI DAN EKSPOR ===
Shape akhir: (130, 7)
Dataset bersih tersimpan di: /content/drive/MyDrive/Colab Notebooks/housing_clean.csv


In [29]:
print("\n=== LANGKAH 11: AKSES API JSONPLACEHOLDER ===")
URL = "https://jsonplaceholder.typicode.com/users"

try:
    response = requests.get(URL, timeout=10)

    # Cek status code sebelum mengakses json [cite: 217]
    if response.status_code == 200:
        data = response.json()
        # Mengonversi respons JSON ke DataFrame Pandas dan melakukan flattening [cite: 26, 209]
        df_users = json_normalize(data, sep='_')
        print("--- [SUKSES] Data API berhasil dikonversi ke DataFrame ---")
        print(df_users[['id', 'name', 'email', 'address_city']].head())
    else:
        print(f"Error: Gagal mengakses API. Status Code: {response.status_code}")

except requests.exceptions.ConnectionError:
    print("Error: Terjadi masalah koneksi saat mengakses API.")


=== LANGKAH 11: AKSES API JSONPLACEHOLDER ===
--- [SUKSES] Data API berhasil dikonversi ke DataFrame ---
   id              name                      email   address_city
0   1     Leanne Graham          Sincere@april.biz    Gwenborough
1   2      Ervin Howell          Shanna@melissa.tv    Wisokyburgh
2   3  Clementine Bauch         Nathan@yesenia.net  McKenziehaven
3   4  Patricia Lebsack  Julianne.OConner@kory.org    South Elvis
4   5  Chelsey Dietrich   Lucio_Hettinger@annie.ca     Roscoeview
